In [1]:
import os, pandas as pd, pandas_gbq
os.chdir(os.path.expanduser("~/growth-marketing-analytics"))
PROJECT_ID = "flash-keel-509517-p3"
sql = open("module_2_google_analytics/sql/01_sessions.sql").read()
print("ready")

ready


In [2]:
ga = pandas_gbq.read_gbq(sql, project_id=PROJECT_ID, dialect="standard")
print(ga.shape)
ga.head()

Downloading: 100%|████████████████████████████████████████████████████████████|
(903653, 18)


,date,visitor_id,visit_id,visit_number,channel,source,medium,campaign,device,country,pageviews,bounce,transactions,revenue,product_view,add_to_cart,checkout,purchase
0,2016-12-08,8522860514027584508,1481197795,4,Organic Search,(direct),(none),(not set),desktop,United States,1,1,0,0.0,False,False,False,False
1,2016-12-08,0337245944735594747,1481197360,1,Direct,(direct),(none),(not set),desktop,India,1,1,0,0.0,False,False,False,False
2,2016-12-08,0395192959157930,1481185793,1,Direct,(direct),(none),(not set),desktop,India,1,1,0,0.0,False,False,False,False
3,2016-12-08,4053165931429228517,1481250098,6,Organic Search,(direct),(none),(not set),mobile,Taiwan,1,1,0,0.0,False,False,False,False
4,2016-12-08,7615170154419369752,1481242074,3,Paid Search,(direct),(none),(not set),mobile,United States,1,1,0,0.0,False,False,False,False


In [3]:
ga.to_parquet("data/processed/ga_sessions.parquet", index=False)
print("saved")

saved


In [4]:
ga["session_id"] = ga.visitor_id + "_" + ga.visit_id.astype(str)
print(f"""
Sessions:          {len(ga):,}
Unique sessions:   {ga.session_id.nunique():,}
Unique visitors:   {ga.visitor_id.nunique():,}
Date range:        {ga.date.min()} to {ga.date.max()}
Transactions:      {ga.transactions.sum():,}
Revenue ($):       {ga.revenue.sum():,.0f}
Channels:          {sorted(ga.channel.unique())}
""")


Sessions:          903,653
Unique sessions:   902,755
Unique visitors:   714,167
Date range:        2016-08-01 to 2017-08-01
Transactions:      12,115
Revenue ($):       1,540,071
Channels:          ['(Other)', 'Affiliates', 'Direct', 'Display', 'Organic Search', 'Paid Search', 'Referral', 'Social']



In [5]:
print("Sessions with revenue but no transaction:", ((ga.revenue > 0) & (ga.transactions == 0)).sum())
print("Sessions with purchase flag:", ga.purchase.sum())
print("Sessions with transactions > 0:", (ga.transactions > 0).sum())
print("Add-to-cart without product view:", (ga.add_to_cart & ~ga.product_view).sum())
print("Purchase without checkout:", (ga.purchase & ~ga.checkout).sum())
print("\nMissing values per column:")
ga.isna().sum()

Sessions with revenue but no transaction: 0
Sessions with purchase flag: 11552
Sessions with transactions > 0: 11552
Add-to-cart without product view: 5612
Purchase without checkout: 10

Missing values per column:


date            0
visitor_id      0
visit_id        0
visit_number    0
channel         0
source          0
medium          0
campaign        0
device          0
country         0
pageviews       0
bounce          0
transactions    0
revenue         0
product_view    0
add_to_cart     0
checkout        0
purchase        0
session_id      0
dtype: int64

In [6]:
checks = pd.Series({
    "sessions": len(ga),
    "unique_visitors": ga.visitor_id.nunique(),
    "transactions": int(ga.transactions.sum()),
    "revenue_usd": round(ga.revenue.sum(), 2),
    "purchase_flag_sessions": int(ga.purchase.sum()),
    "sessions_with_transactions": int((ga.transactions > 0).sum()),
    "cart_without_product_view": int((ga.add_to_cart & ~ga.product_view).sum()),
})
checks.to_csv("module_2_google_analytics/outputs/data_checks.csv")
checks

sessions                       903653.00
unique_visitors                714167.00
transactions                    12115.00
revenue_usd                   1540071.24
purchase_flag_sessions          11552.00
sessions_with_transactions      11552.00
cart_without_product_view        5612.00
dtype: float64